在工具内部访问运行时信息（如对话历史、用户数据和持久化记忆）

#### 工具内部访问 State（短期记忆）

访问对话历史，跟踪工具调用次数

In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langchain.agents import create_agent, AgentState
from langgraph.types import Command
from langchain.chat_models import init_chat_model
from agent_lab.model_hub import LLM_FAST

@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the user."""
    messages = runtime.state["messages"]

    # Find the last human message
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return message.content # type: ignore

    return "No user messages found"

# Access custom state fields
@tool
def get_user_preference(
    pref_name: str,
    runtime: ToolRuntime
) -> str:
    """Get a user preference value."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")

@tool
def set_user_name(new_name: str, runtime: ToolRuntime) -> Command:
    """Set the user's name in the conversation state."""
    return Command(
        update={
            "user_name": new_name,
            "messages": [
                ToolMessage(
                    content=f'user name updated to: {new_name}',
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

class CustomState(AgentState):
    user_preferences: dict
    user_name: str

agent = create_agent(
    model=init_chat_model(**LLM_FAST),
    tools=[get_last_user_message, get_user_preference, set_user_name],
    state_schema=CustomState,
)

In [2]:
state = {
    'messages': [HumanMessage('What are my preferences.')],
    'user_preferences': {'style': 'technical', 'verbosity': 'detailed'},
}
state = agent.invoke(state) # type: ignore

In [3]:
for msg in state['messages']:
    msg.pretty_print()

================================ Human Message =================================

What are my preferences.
================================== Ai Message ==================================

Let me check your preferences for you.
Tool Calls:
  get_user_preference (call_00_U83BgHbxa2nK9AnKrqqAoyvF)
 Call ID: call_00_U83BgHbxa2nK9AnKrqqAoyvF
  Args:
    pref_name: *
================================= Tool Message =================================
Name: get_user_preference

Not set
================================== Ai Message ==================================

It appears that no preferences have been saved for you yet. There are no current preferences set in the system. 

Would you like to set some preferences? If so, let me know what you'd like to save, and I can help you with that!


In [4]:
state['messages'].append(
    HumanMessage('Set the username to `MorePeanuts`')
)
state = agent.invoke(state) # type: ignore

In [5]:
for msg in state['messages'][4:]:
    msg.pretty_print()

================================ Human Message =================================

Set the username to `MorePeanuts`
================================== Ai Message ==================================
Tool Calls:
  set_user_name (call_00_dj6E5aHeTmYTbwsaThDnk4YG)
 Call ID: call_00_dj6E5aHeTmYTbwsaThDnk4YG
  Args:
    new_name: MorePeanuts
================================= Tool Message =================================
Name: set_user_name

user name updated to: MorePeanuts
================================== Ai Message ==================================

Done! Your username has been set to **MorePeanuts**. 

Is there anything else you'd like to set or check?


In [9]:
print(state['user_name'])

MorePeanuts


In [10]:
state['messages'].append(
    HumanMessage('What was the last human message?')
)
state = agent.invoke(state) # type: ignore

In [11]:
for msg in state['messages'][8:]:
    msg.pretty_print()

================================ Human Message =================================

What was the last human message?
================================== Ai Message ==================================
Tool Calls:
  get_last_user_message (call_00_NnPZ2Hwz9gR7kVLANupo1PJo)
 Call ID: call_00_NnPZ2Hwz9gR7kVLANupo1PJo
  Args:
================================= Tool Message =================================
Name: get_last_user_message

What was the last human message?
================================== Ai Message ==================================

The last human message was exactly that question: **"What was the last human message?"**
